# 09 - Combined Results and Final Summary

This notebook loads all experiment outputs, builds unified comparison tables, generates summary visualizations, and prints the final experiment summary block.

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
import sys
sys.modules["tensorflow"] = None

### What this does and why

Reviewer-friendly reporting needs one consolidated view. This notebook combines all sub-experiment outputs into final tables and figures suitable for presentation.

In [2]:
from pathlib import Path
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

root = Path.cwd()
outputs_dir = root / "outputs"
if not outputs_dir.exists():
    outputs_dir = root.parent / "outputs"

s1 = pd.read_csv(outputs_dir / "zeroshot_vs_trained_comparison.csv")
s2 = pd.read_csv(outputs_dir / "prompt_engineering_results.csv")
s3 = pd.read_csv(outputs_dir / "fewshot_vs_zeroshot.csv")
s4 = pd.read_csv(outputs_dir / "error_analysis_summary.csv")
basic_metrics = json.loads((outputs_dir / "zeroshot_basic_metrics.json").read_text(encoding="utf-8"))
few_meta = json.loads((outputs_dir / "fewshot_vs_zeroshot_meta.json").read_text(encoding="utf-8"))

best_trained_row = s1[s1["Type"] == "Trained"].sort_values("Weighted F1", ascending=False).iloc[0]
best_prompt_row = s2.sort_values("Weighted F1", ascending=False).iloc[0]
zero_row = s3[s3["Method"] == "Zero-Shot"].iloc[0]
few_row = s3[s3["Method"] == "Few-Shot"].iloc[0]

combined = []
for _, r in s1.iterrows():
    combined.append({
        "Section": "Sub-exp 1",
        "Item": r["Model"],
        "Type": r["Type"],
        "Accuracy": r["Accuracy"],
        "Weighted F1": r["Weighted F1"],
        "Macro F1": r["Macro F1"],
    })
for _, r in s2.iterrows():
    combined.append({
        "Section": "Sub-exp 2",
        "Item": f"Prompt {r['Prompt']} ({r['Labels Used']})",
        "Type": "Prompt Variant",
        "Accuracy": r["Accuracy"],
        "Weighted F1": r["Weighted F1"],
        "Macro F1": r["Macro F1"],
    })
for _, r in s3.iterrows():
    combined.append({
        "Section": "Sub-exp 3",
        "Item": r["Method"],
        "Type": "Inference Method",
        "Accuracy": r["Accuracy"],
        "Weighted F1": r["Weighted F1"],
        "Macro F1": r["Macro F1"],
    })

combined_df = pd.DataFrame(combined)
combined_df.to_csv(outputs_dir / "combined_experiment_comparison.csv", index=False)

plt.figure(figsize=(10, 4))
plot_df = combined_df[combined_df["Section"].isin(["Sub-exp 1", "Sub-exp 3"])]
sns.barplot(data=plot_df, x="Item", y="Weighted F1", hue="Section")
plt.ylim(0, 1)
plt.title("Final Weighted F1 Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(outputs_dir / "final_weighted_f1_summary.png", dpi=220)
plt.close()

plt.figure(figsize=(7, 4))
sns.barplot(data=s4, x="category", y="pct")
plt.title("Error Analysis Category Distribution")
plt.ylabel("Percentage")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(outputs_dir / "final_error_category_distribution.png", dpi=220)
plt.close()

summary_map = {r['category']: (int(r['count']), float(r['pct'])) for _, r in s4.iterrows()}
n_llm, p_llm = summary_map.get("LLM advantage", (0, 0.0))
n_cls, p_cls = summary_map.get("Classical advantage", (0, 0.0))
n_ag, p_ag = summary_map.get("Agreement correct", (0, 0.0))
n_h, p_h = summary_map.get("Hard cases", (0, 0.0))

trained_f1 = float(best_trained_row["Weighted F1"])
zs_basic_f1 = float(basic_metrics["f1_weighted"])
gap = trained_f1 - zs_basic_f1
gap_winner = "trained wins" if gap > 0 else "LLM wins"

print("=== EXPERIMENT SUMMARY ===\n")
print("Sub-exp 1: Zero-Shot vs Trained Models")
print(f"  Best trained model: {best_trained_row['Model']} - Weighted F1: {trained_f1:.4f}")
print(f"  Zero-shot BART:              Weighted F1: {zs_basic_f1:.4f}")
print(f"  Gap: {gap:.4f} ({gap_winner})\n")

print("Sub-exp 2: Best Prompt")
print(f"  Best prompt: {best_prompt_row['Prompt']}")
print(f"  Labels: {[best_prompt_row['label_0_text'], best_prompt_row['label_1_text']]}")
print(f"  Weighted F1: {float(best_prompt_row['Weighted F1']):.4f}\n")

print("Sub-exp 3: Few-Shot vs Zero-Shot")
print(f"  Zero-shot F1: {float(zero_row['Weighted F1']):.4f}")
print(f"  Few-shot F1:  {float(few_row['Weighted F1']):.4f}")
print(f"  Winner: {few_meta['winner']}\n")

print("Sub-exp 4: Error Analysis")
print(f"  LLM advantage cases:    {n_llm} ({p_llm:.1f}%)")
print(f"  Classical advantage:    {n_cls} ({p_cls:.1f}%)")
print(f"  Both correct:           {n_ag} ({p_ag:.1f}%)")
print(f"  Hard cases (both wrong):{n_h} ({p_h:.1f}%)\n")

key_finding = "Label wording and contextual prompting can improve LLM behavior, but the best classical model remains stronger overall on this Reddit risk task."
print("Key Finding:", key_finding)

combined_df.head(12)

=== EXPERIMENT SUMMARY ===

Sub-exp 1: Zero-Shot vs Trained Models
  Best trained model: SVM - Weighted F1: 0.7887
  Zero-shot BART:              Weighted F1: 0.5459
  Gap: 0.2428 (trained wins)

Sub-exp 2: Best Prompt
  Best prompt: B
  Labels: ['general depression anxiety and loneliness', 'suicidal ideation and self-harm']
  Weighted F1: 0.7033

Sub-exp 3: Few-Shot vs Zero-Shot
  Zero-shot F1: 0.7033
  Few-shot F1:  0.6028
  Winner: Zero-Shot

Sub-exp 4: Error Analysis
  LLM advantage cases:    10 (25.0%)
  Classical advantage:    7 (17.5%)
  Both correct:           21 (52.5%)
  Hard cases (both wrong):2 (5.0%)

Key Finding: Label wording and contextual prompting can improve LLM behavior, but the best classical model remains stronger overall on this Reddit risk task.


,Section,Item,Type,Accuracy,Weighted F1,Macro F1
0,Sub-exp 1,SVM,Trained,0.78875,0.788726,0.788726
1,Sub-exp 1,MLP,Trained,0.76400,0.763938,0.763938
2,Sub-exp 1,LR,Trained,0.75725,0.757150,0.757150
3,Sub-exp 1,BART-ZS,Zero-Shot,0.56500,0.545917,0.545917
4,Sub-exp 2,Prompt B (descriptive),Prompt Variant,0.71000,0.703325,0.703325
5,Sub-exp 2,Prompt A (simple),Prompt Variant,0.68500,0.657152,0.657152
6,Sub-exp 2,Prompt C (clinical),Prompt Variant,0.53500,0.413601,0.413601
7,Sub-exp 3,Zero-Shot,Inference Method,0.71000,0.703325,0.703325
8,Sub-exp 3,Few-Shot,Inference Method,0.60500,0.602766,0.602766
